# YouTube Playlist Transcripts (keyword filter)

Give it a playlist link and a keyword. It pulls every video in the playlist whose title contains the keyword,
grabs details and transcripts, and writes the results to a Google Sheet tab.

**Setup:** add `YOUTUBE_API_KEY` in Colab's Secrets panel (the key icon on the left), then run all cells.

Rerunning is safe: videos that already have a transcript in the sheet are skipped, so only missing or failed ones are retried.

In [ ]:
!pip -q install --upgrade google-api-python-client youtube-transcript-api gspread gspread-dataframe pandas

## Settings

In [ ]:
# Paste a playlist URL or a bare playlist ID
PLAYLIST = "https://youtube.com/playlist?list=PLpZimxTCBwRV3_2LODwiiicfukWaV3VhT"

# Only keep videos whose title contains this text (case-insensitive). Set to "" to keep everything.
TITLE_KEYWORD = "Claude"

SPREADSHEET_ID = "1DqnXgFMetI2FMc_6dYYQr3cIKHPU-ee07KXSQdPVfuQ"
WORKSHEET_NAME = "Claude"   # tab is created if it doesn't exist

TRANSCRIPT_LANGS = ["en", "en-US", "en-GB"]
SLEEP_SECONDS_BETWEEN_TRANSCRIPTS = 1.0
CHECKPOINT_EVERY = 10       # write to the sheet after this many transcripts

## Auth

In [ ]:
from google.colab import auth, userdata
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

YOUTUBE_API_KEY = userdata.get("YOUTUBE_API_KEY")

## Helpers

In [ ]:
import random, re, time
from urllib.parse import urlparse, parse_qs

import pandas as pd
from googleapiclient.discovery import build
from gspread_dataframe import get_as_dataframe, set_with_dataframe
from youtube_transcript_api import NoTranscriptFound, YouTubeTranscriptApi

youtube = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)
ytt = YouTubeTranscriptApi()

SHEETS_CELL_LIMIT = 50000  # Google Sheets max characters per cell


def parse_playlist_id(playlist: str) -> str:
    """Accepts a full playlist URL (any youtube domain) or a bare ID."""
    s = (playlist or "").strip()
    if "list=" in s:
        qs = parse_qs(urlparse(s).query)
        if qs.get("list"):
            return qs["list"][0]
    return s


def title_matches(title: str, keyword: str) -> bool:
    return (not keyword) or keyword.lower() in (title or "").lower()


def fetch_playlist_items(playlist_id: str):
    """All videos in the playlist, in playlist order."""
    items, token = [], None
    while True:
        resp = youtube.playlistItems().list(
            part="snippet,contentDetails",
            playlistId=playlist_id,
            maxResults=50,
            pageToken=token,
        ).execute()
        for it in resp.get("items", []):
            snip = it.get("snippet", {})
            vid = it.get("contentDetails", {}).get("videoId")
            items.append({
                "Playlist Position": len(items) + 1,
                "Video ID": vid,
                "Title": snip.get("title", ""),
            })
        token = resp.get("nextPageToken")
        if not token:
            return items


def chunk(lst, n=50):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]


def fetch_video_details_map(video_ids):
    """videoId -> videos.list item (snippet, contentDetails, statistics)."""
    out = {}
    for ids in chunk([v for v in video_ids if v], 50):
        resp = youtube.videos().list(
            part="snippet,contentDetails,statistics",
            id=",".join(ids),
            maxResults=50,
        ).execute()
        for v in resp.get("items", []):
            out[v["id"]] = v
    return out


def iso8601_duration_to_seconds(iso: str) -> int:
    m = re.match(r"^PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?$", iso or "")
    if not m:
        return 0
    h, mn, s = (int(g or 0) for g in m.groups())
    return h * 3600 + mn * 60 + s


def seconds_to_hms(sec: int) -> str:
    sec = int(sec or 0)
    h, m, s = sec // 3600, (sec % 3600) // 60, sec % 60
    return f"{h}:{m:02d}:{s:02d}" if h else f"{m}:{s:02d}"

## Transcript fetch with retry

In [ ]:
BLOCK_KEYWORDS = ("blocking requests from your ip", "requestblocked", "ipblocked", "toomanyrequests", "429")
NO_TRANSCRIPT_KEYWORDS = ("transcripts are disabled", "notranscriptfound", "no transcripts", "video is unavailable")


def is_block_error(msg: str) -> bool:
    msg = (msg or "").lower()
    return any(k in msg for k in BLOCK_KEYWORDS)


def fetch_transcript_once(video_id: str) -> str:
    try:
        fetched = ytt.fetch(video_id, languages=TRANSCRIPT_LANGS)
    except NoTranscriptFound:
        # No English track: take whatever language the video has
        fetched = next(iter(ytt.list(video_id))).fetch()
    return "\n".join(s.text for s in fetched)


def fetch_transcript_with_retry(video_id: str, max_attempts=6, base_sleep=2.0, max_sleep=180.0) -> str:
    last_err = ""
    for attempt in range(1, max_attempts + 1):
        try:
            print(f"[{attempt}/{max_attempts}] {video_id}")
            text = fetch_transcript_once(video_id)
            if text.strip():
                return text
            last_err = "Empty transcript response"
        except Exception as exc:
            last_err = f"{type(exc).__name__}: {exc}"
            if not is_block_error(last_err) and any(k in last_err.lower() for k in NO_TRANSCRIPT_KEYWORDS):
                return f"ERROR: {last_err}"

        block = is_block_error(last_err)
        sleep = min(max_sleep, base_sleep * (3.0 if block else 1.0) * 2 ** (attempt - 1))
        sleep *= 0.6 + random.random() * 0.8  # jitter
        print(f"   {'BLOCKED' if block else 'retry'}: sleeping {sleep:.1f}s | {last_err[:120]}")
        time.sleep(sleep)
    return f"ERROR: {last_err}"


def needs_transcript(val) -> bool:
    v = "" if val is None or (isinstance(val, float) and pd.isna(val)) else str(val).strip()
    return v == "" or v.startswith("ERROR:")

## Build the video list

In [ ]:
playlist_id = parse_playlist_id(PLAYLIST)
all_items = fetch_playlist_items(playlist_id)
matches = [it for it in all_items if it["Video ID"] and title_matches(it["Title"], TITLE_KEYWORD)]
print(f"Playlist {playlist_id}: {len(all_items)} videos, {len(matches)} match '{TITLE_KEYWORD}'")

details = fetch_video_details_map([it["Video ID"] for it in matches])

rows = []
for it in matches:
    vid = it["Video ID"]
    vd = details.get(vid, {})
    snip, stats, cdet = vd.get("snippet", {}), vd.get("statistics", {}), vd.get("contentDetails", {})
    secs = iso8601_duration_to_seconds(cdet.get("duration", ""))
    tags = snip.get("tags", [])
    rows.append({
        "Playlist Position": it["Playlist Position"],
        "Video ID": vid,
        "Title": it["Title"],
        "URL": f"https://www.youtube.com/watch?v={vid}",
        "Channel": snip.get("channelTitle", ""),
        "Published At": snip.get("publishedAt", ""),
        "Duration (hh:mm:ss)": seconds_to_hms(secs) if secs else "",
        "Duration Seconds": secs,
        "Views": int(stats.get("viewCount", 0) or 0),
        "Likes": int(stats["likeCount"]) if "likeCount" in stats else "",
        "Comments": int(stats["commentCount"]) if "commentCount" in stats else "",
        "Tags": " | ".join(tags) if isinstance(tags, list) else "",
        "Description": snip.get("description", "") or "",
        "Transcript": "",
        "Summary": "",
    })

df = pd.DataFrame(rows)
df[["Playlist Position", "Title", "Duration (hh:mm:ss)"]]

## Reuse transcripts already in the sheet

In [ ]:
sh = gc.open_by_key(SPREADSHEET_ID)
try:
    ws = sh.worksheet(WORKSHEET_NAME)
except gspread.WorksheetNotFound:
    ws = sh.add_worksheet(title=WORKSHEET_NAME, rows=100, cols=len(df.columns))

existing = get_as_dataframe(ws, evaluate_formulas=False).dropna(how="all")
if not existing.empty and {"Video ID", "Transcript"} <= set(existing.columns):
    for col in ("Transcript", "Summary"):
        if col in existing.columns:
            old = existing.dropna(subset=["Video ID"]).set_index("Video ID")[col]
            df[col] = df["Video ID"].map(old).fillna("").astype(str)

todo = df.index[df["Transcript"].apply(needs_transcript)]
print(f"Need transcripts for {len(todo)} of {len(df)} videos")

## Fetch transcripts and write to the sheet

In [ ]:
def write_sheet():
    out = df.copy()
    # Sheets rejects cells over 50k chars; keep the start of long transcripts
    out["Transcript"] = out["Transcript"].astype(str).apply(
        lambda t: t if len(t) <= SHEETS_CELL_LIMIT else t[:SHEETS_CELL_LIMIT - 20] + "\n[TRUNCATED]"
    )
    ws.clear()
    set_with_dataframe(ws, out, include_index=False, include_column_header=True, resize=True)


write_sheet()
for n, idx in enumerate(todo, 1):
    df.at[idx, "Transcript"] = fetch_transcript_with_retry(df.at[idx, "Video ID"])
    time.sleep(SLEEP_SECONDS_BETWEEN_TRANSCRIPTS)
    if n % CHECKPOINT_EVERY == 0:
        print(f"Checkpoint ({n}/{len(todo)})")
        write_sheet()

write_sheet()
failed = df["Transcript"].apply(needs_transcript).sum()
print(f"Done: {len(df)} videos written to '{WORKSHEET_NAME}', {failed} without a transcript")